In [1]:
import numpy as np
from scipy.special import softmax
# import pandas as pd

from rlenvs import Hunting 

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim
import pandas as pd

In [2]:
class Estimator(nn.Sequential):
    def __init__(self, input_shape:tuple[int], output_size:int, lr=0.01):
        super(Estimator, self).__init__(
            nn.Linear(np.sum(input_shape), output_size)
        )
        self.input_shape = input_shape
        self.opt = optim.Adam(self.parameters(), lr=lr)

    def train(self, X, Y, num_epochs=10):
        inputs_values = torch.cat([torch.nn.functional.one_hot(torch.tensor(value), num_classes=size) for value, size in zip(X, self.input_shape)]).float()
        Y = torch.tensor(Y).float()
        for i in range(num_epochs):
            self.opt.zero_grad()
            pred = self(inputs_values)
            loss = f.mse_loss(pred, Y)
            loss.backward()
            self.opt.step()  

    def predict(self, *x):
        with torch.no_grad():
            inputs_values = torch.cat([
                torch.nn.functional.one_hot(
                    torch.tensor(value), 
                    num_classes=size
                ) 
                for value, size in zip(x, self.input_shape)
            ])
            logits = self.forward(inputs_values.float())
        return logits

est = Estimator([2,3], 1)
# est(torch.tensor([[1.,0., 0., 0., 0.]]))
est.predict(1,0)

tensor([-0.4956])

In [3]:
class Estimator_T(Estimator):    
    def __init__(self, input_shape:tuple[int], hidden_size:int, output_size:int, lr=0.01):
        super(Estimator_T, self).__init__(input_shape, output_size, lr)
        super(Estimator, self).__init__(
            nn.Linear(np.sum(input_shape), hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
            # nn.Softmax()
        )

    def train(self, X, Y, num_epochs=10):
        inputs_values = torch.cat([torch.nn.functional.one_hot(torch.tensor(value), num_classes=size) for value, size in zip(X, self.input_shape)]).float()
        Y = torch.tensor(Y).long()
        for i in range(num_epochs):
            self.opt.zero_grad()
            pred = self(inputs_values)
            loss = f.cross_entropy(pred, Y)
            loss.backward()
            self.opt.step() 

    def proba(self, *x):
       logit = self.predict(*x)
       return f.softmax(logit, dim=0)

In [4]:
class Model:
    def __init__(self, S,A, rho=.9):
        self.S = S
        self.A = A
        self.N = 0
        self.rho = rho
        self._E = 0 

        self.t = Estimator_T((len(self.S), len(self.A)), 100, len(self.S))
        self.r = Estimator((len(self.S), len(self.A), len(self.S)), 1)

    def e(self, s,a,s_,r):
        pred_s = np.argmax(self.t.proba(s,a))
        pred_r = self.r.predict(s,a, pred_s)
        err = np.sqrt(np.pow(pred_s - s_, 2)) + np.sqrt(np.pow(pred_r - r, 2))
        return err
    
    def E(self, s,a,s_,r):
        self._E += self.rho*(self.e(s,a,s_,r) - self._E)
        return self._E

    def learn(self, s,a,s_,r):
        self.N += 1 
        self.t.train([s,a],s_)
        self.r.train([s,a,s_],r)
    
    def simulate(self, s, a): 
        s_ = np.random.choice(len(self.S), p=self.t.proba(s,a).numpy())
        r = self.r.predict(s,a,s_)
        return s_,r
        
    def T(self, s,a,s_): # predict_T
        pred = self.t.predict(s,a)[s_]
        return pred
    def R(self, s,a,s_):
        pred = self.r.predict(s,a,s_)
        return pred

model = Model(range(10), range(4))

In [5]:
class KModels:
    def __init__(self, S,A, k=4, Emin=-0.01, M=1e2):
        self.S = S
        self.A = A
        self.current_model = None
        self.k = k
        self.Emin = Emin
        self.M = M
        self.Models = []
        for _ in range(self.k):
            self.new_model()

    def new_model(self, M=1e2):
        self.current_model = Model(self.S, self.A)
        self.Models.append(self.current_model)

    def learn(self, s,a,s_,r, log=False):
        E = [m.E(s,a,s_,r) for m in self.Models]
        self.current_model = self.Models[ np.argmax(E) ]
        # new_model = self.current_model._E < self.Emin
        
        # if log:
        #     print('E: ', E, new_model)
        # if new_model:
        #     self.new_model()

        self.current_model.learn(s,a,s_,r)
    
    def simulate(self, s, a):
        sims = [m.simulate(s,a) for m in self.Models]
        E = [m.E(s,a,s_,r) for m, (s_,r) in zip(self.Models, sims)]
        return sims[ np.argmax(E) ]


model = KModels(range(10), range(4))

# Algorítimo de Controle (RL):
 - Dyna Architecture

In [6]:
class Dyna:
    def __init__(self, model, n=100, alpha=.9, gamma=.9, epsilon=.1):
        self.model = model
        self.n = n
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = np.zeros((self.model.S.size, self.model.A.size))
        self.hist = np.zeros((self.model.S.size, self.model.A.size))

    def run(self,s,a):
        self.hist[s,a] += 1

        Ss = np.random.choice([i for i,v in enumerate(np.sum(self.hist, axis=1)) if v>0], self.n) # n random seen states 
        for s in Ss:
            a = np.random.choice([i for i,v in enumerate(self.hist[s]) if v>0]) # a random taken action in state s
            s_,r = self.model.simulate(s,a)
            self.Q[s,a] += self.alpha*(r + self.gamma*np.max(self.Q[s_]) - self.Q[s,a]) 

    def v(self):
        return np.max(self.Q, axis=1)
    def control(self):
        return np.argmax(self.Q, axis=1)

# Agente

In [7]:
class Agent:
    def __init__(self, S, A, n=100, alpha=.9 ,gamma=.9, epsilon=.1):
        self.n = n
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.model = KModels(S, A)
        self.rl = Dyna(self.model, self.n, self.alpha, self.gamma, self.epsilon)
    
    def learn(self, s,a,s_,r):
        self.model.learn(s,a,s_,r)
        self.rl.run(s,a)

    def act(self, s):
        pi = self.get_policy()
        return pi[s]
    def evaluate(self, s):
        v = self.get_v()
        return v[s]
    
    def get_v(self):
        return self.rl.v()
    def get_policy(self):
        return self.rl.control()
                

# Simulação

In [8]:
def generate_episode(env,agent, size_limit=100):
    data = []
    env.reset()
    for _ in range(size_limit):
        s, _, _, _, _ = env.last()
        a = agent.act(s)
        s_, r, end, _, _ = env.step(s,a)
        
        step = (s,a,s_,r) 
        agent.learn(*step) 
        data.append(step)
        if end: break
    return data

def experiment(env, agent, max_iterations=1000, episode_sizes=100):
    data = []
    for _ in range(max_iterations):
        epi = generate_episode(env, agent, episode_sizes)
        data.append(len(epi))    
    return data



# Experimentação

In [9]:
env = Hunting(5,5)

n = 100
alpha = 0.9
epsilon = 0.1
gamma = 0.9
num_episodes = 100
episode_size = 12

agent = Agent(env.S, env.A, n, alpha, gamma, epsilon)

In [10]:
exp = experiment(env, agent, num_episodes, episode_size)
# exp

/tmp/ipykernel_9782/39381975.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(value),
/tmp/ipykernel_9782/1283577064.py:15: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  err = np.sqrt(np.pow(pred_s - s_, 2)) + np.sqrt(np.pow(pred_r - r, 2))
/tmp/ipykernel_9782/39381975.py:15: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = f.mse_loss(pred, Y)


In [11]:
env.plot(list(np.round(agent.get_v(), 2)))
env.plot(list(agent.get_policy()), True)

 _____________________________ 
|-0.72|-0.64|-0.91| 0.0 | 0.0 |
|_____|_____|_____|_____|_____|
| 0.0 |-0.62|-0.67| 0.0 | 0.0 |
|_____|_____|_____|_____|_____|
|-0.69|-0.53| 0.0 |-0.67| 0.0 |
|_____|_____|_____|_____|_____|
|-0.85| 0.0 |-0.68| 0.0 | 0.0 |
|_____|_____|_____|_____|_____|
| 0.0 |-0.61|-0.77| 0.0 | 0.0 |
|_____|_____|_____|_____|_____|

 _____________________________ 
|  ←  |  ↓  |  ←  |  ←  |  ←  |
|_____|_____|_____|_____|_____|
|  ←  |  ←  |  ↓  |  ←  |  →  |
|_____|_____|_____|_____|_____|
|  ←  |  ↓  |  ←  |  ↑  |  ←  |
|_____|_____|_____|_____|_____|
|  ↓  |  →  |  ↓  |  ←  |  ↓  |
|_____|_____|_____|_____|_____|
|  ↑  |  ←  |  ←  |  ↓  |  ←  |
|_____|_____|_____|_____|_____|

